# Notebook 1 — Definição do Problema e Coleta de Dados
**Projeto:** Predição de Evasão Escolar em Pernambuco  
**Disciplina:** Aprendizagem de Máquina | **Entrega:** 11/05/2026
---

## 1. Contexto

A evasão escolar é um dos principais desafios da educação pública brasileira. Em Pernambuco, desigualdades regionais, diferenças entre redes de ensino e condições de infraestrutura escolar agravam esse fenômeno. A identificação de **onde** o risco é maior — e quais condições estruturais estão associadas ao abandono — pode apoiar gestores públicos na formulação de intervenções mais eficazes.

## 2. Objetivo Geral

Desenvolver um pipeline de Machine Learning capaz de classificar contextos educacionais quanto ao **risco de evasão escolar**, integrando taxas de rendimento escolar por município com indicadores de infraestrutura das escolas do Brasil — cobrindo os **5.570 municípios** brasileiros na granularidade de município × localização × dependência administrativa.

## 3. Objetivos Específicos

1. Integrar taxas de rendimento escolar com dados de infraestrutura na granularidade de **município × localização × dependência administrativa** (5.570 municípios)
2. Analisar como localização (urbano/rural) e dependência administrativa se associam ao abandono  
3. Investigar quais recursos de infraestrutura se correlacionam com menor evasão  
4. Construir e comparar modelos de classificação binária supervisionada  
5. Interpretar os resultados com foco em impacto para políticas públicas em Pernambuco

## 4. Tipo de Problema

**Classificação binária supervisionada.**  
A variável-alvo indica se um dado contexto (**município × localização × dependência administrativa**) apresenta **alto risco de evasão** (1) ou **baixo risco** (0), definida a partir do percentil 75 nacional da taxa de abandono geral (~2,05%).

## 5. Hipóteses

| # | Hipótese |
|---|----------|
| H1 | Maior taxa de reprovação está associada a maior risco de evasão |
| H2 | Escolas rurais apresentam maior risco de abandono que escolas urbanas |
| H3 | Escolas municipais têm risco maior que escolas estaduais e privadas |
| H4 | O Ensino Médio tem risco de evasão maior que o Ensino Fundamental |
| H5 | Contextos com menor acesso à internet e biblioteca têm maior abandono |
| H6 | Escolas municipais rurais concentram o maior risco de evasão |

## 6. Critérios de Sucesso

| Critério | Meta |
|----------|------|
| ROC-AUC | ≥ 0.70 |
| Recall (classe positiva) | Maximizar — custo de falso negativo é maior que falso positivo |
| Interpretabilidade | Identificar variáveis mais relevantes via feature importance ou SHAP |
| Modelo final | Superar DummyClassifier em todas as métricas |

## 7. Fontes de Dados

### Dataset 1 — Taxas de Rendimento por Município *(modelagem principal)*

| Item | Detalhe |
|------|---------|
| Arquivo | `tx_rend_municipios_2024.xlsx` |
| Fonte | INEP — Instituto Nacional de Estudos e Pesquisas Educacionais Anísio Teixeira |
| URL | https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/indicadores-educacionais/taxas-de-rendimento-escolar |
| Cobertura | 5.570 municípios do Brasil — 2024 |
| Granularidade | Município × Localização × Dependência administrativa |
| Instâncias (bruto) | 65.594 linhas × 61 colunas |
| Instâncias (após limpeza) | 50.197 linhas × indicadores de reprovação e abandono |
| Municípios de PE | 185 municípios |
| Licença | Dados abertos do governo federal |

### Dataset 2 — Microdados de Infraestrutura Escolar

| Item | Detalhe |
|------|---------|
| Arquivo | `microdados_ed_basica_2024.csv` |
| Fonte | INEP — Censo Escolar da Educação Básica 2024 |
| URL | https://www.gov.br/inep/pt-br/acesso-a-informacao/dados-abertos/microdados/censo-escolar |
| Cobertura | Todas as escolas do Brasil — 215.545 escolas |
| Tamanho bruto | 208 MB, 426 colunas por escola |
| Colunas selecionadas | 23 colunas de infraestrutura + localização + dependência + município |
| Granularidade original | Por escola → **agregado** por Município × Localização × Dependência |
| Resultado agregado | 53.497 linhas × 22 indicadores de infraestrutura |
| Licença | Dados abertos do governo federal |

## 8. Estratégia de Integração

Os dois datasets compartilham as chaves `cod_municipio × localizacao × dependencia_adm`. Os microdados são **agregados por município**, calculando o percentual de escolas com cada recurso de infraestrutura em cada grupo, produzindo uma tabela compatível. O join é feito por essas três chaves com `merge left`.

```
Dataset 1 (rendimento municipal)       Dataset 2 (infraestrutura agregada)
Município × Localização × Dep.  →merge← Município × Localização × Dep.
reprovação, abandono...                  % internet, % biblioteca...
50.197 linhas                            53.497 linhas
                      ↓
              50.140 instâncias completas
         (5.570 municípios × ~9 combinações)
```

A variável-alvo `risco_evasao` é construída após o merge: contextos com `abandono_geral` acima do **P75 nacional (≈ 2,05%)** recebem rótulo 1 (alto risco), os demais recebem 0.

## 9. Dicionário de Dados — Dataset Final Integrado

**Shape:** 50.140 instâncias × 84 colunas  
**Chave de cada instância:** município × localização × dependência administrativa

### Variáveis de identificação (5 colunas)
| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `cod_municipio` | str | Código IBGE do município (7 dígitos) |
| `nome_municipio` | str | Nome do município |
| `uf` | str | Sigla do estado (ex: PE, SP) |
| `localizacao` | str | Total, Urbana ou Rural |
| `dependencia_adm` | str | Total, Federal, Estadual, Municipal ou Privada |

### Taxas de rendimento — Dataset 1 (14 colunas usadas)
| Coluna | Descrição | Usado no modelo |
|--------|-----------|-----------------|
| `reprov_fund_total` | Reprovação — EF total (%) | ✓ feature |
| `reprov_fund_anos_iniciais` / `_anos_finais` | Reprovação por etapa EF (%) | `_anos_finais` ✓ |
| `reprov_med_total` | Reprovação — EM total (%) | ✓ feature |
| `reprov_med_1serie` / `_2serie` / `_3serie` | Reprovação por série do EM (%) | `_1serie` ✓ |
| `abandono_fund_total` / `_anos_finais` | Abandono — EF (%) | base do alvo |
| `abandono_med_total` / `_1serie` / `_2serie` / `_3serie` | Abandono — EM (%) | base do alvo |
| `abandono_geral` | Média `abandono_fund_total` + `abandono_med_total` — **construída** (%) | base do alvo |
| `risco_evasao` | **Variável-alvo**: 1 = `abandono_geral` > P75 (≈ 2,05%), 0 = baixo risco | alvo |

> **Nota:** colunas de aprovação (`aprov_*`) não entram no modelo — são matematicamente derivadas de reprovação + abandono (soma = 100%), o que causaria leakage indireto.

### Indicadores de infraestrutura — Dataset 2 agregado (21 colunas)
| Coluna | Descrição |
|--------|-----------|
| `pct_internet` | % de escolas com acesso à internet |
| `pct_internet_alunos` | % de escolas com internet disponível para alunos |
| `pct_banda_larga` | % de escolas com banda larga |
| `pct_biblioteca` | % de escolas com biblioteca |
| `pct_sala_leitura` | % de escolas com sala de leitura |
| `pct_lab_ciencias` | % de escolas com laboratório de ciências |
| `pct_lab_informatica` | % de escolas com laboratório de informática |
| `pct_quadra` | % de escolas com quadra esportiva |
| `pct_energia_rede` | % de escolas com energia elétrica da rede pública |
| `pct_sem_energia` | % de escolas sem nenhuma fonte de energia |
| `pct_agua_potavel` | % de escolas com água potável |
| `pct_sem_agua` | % de escolas sem acesso à água |
| `pct_esgoto_rede` | % de escolas com esgoto da rede pública |
| `pct_sem_esgoto` | % de escolas sem sistema de esgoto |
| `pct_banheiro` | % de escolas com banheiro |
| `pct_banheiro_acessivel` | % de escolas com banheiro acessível (PNE) |
| `pct_alimentacao` | % de escolas com alimentação escolar |
| `pct_sem_acessibilidade` | % de escolas sem nenhum recurso de acessibilidade |
| `pct_computador` | % de escolas com computador |
| `qt_salas_media` | Média de salas utilizadas por escola |
| `n_escolas` | Número de escolas no grupo |

## 10. Carregamento Inicial — Verificação das Fontes

In [5]:
from pathlib import Path
import os

# Garante que o diretório de trabalho é sempre a raiz do projeto
PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == 'notebooks':
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
print(f'Diretório de trabalho: {PROJECT_ROOT}')

Diretório de trabalho: /Users/joaogui/Downloads/1VA_Aprendizado_Maquina


In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# ── Dataset principal: Taxas de Rendimento por Município ─────────────────
df_raw = pd.read_excel(
    'data/raw/tx_rend_municipios_2024.xlsx',
    sheet_name='MUNICIPIOS ', header=None, skiprows=8
)
df_raw.columns = df_raw.iloc[0].tolist()
df_rend = df_raw.iloc[1:].copy().reset_index(drop=True)
df_rend = df_rend[
    df_rend['NU_ANO_CENSO'].notna() &
    (df_rend['NU_ANO_CENSO'] != 'Fonte: INEP/Censo Escolar da Educação Básica.')
].copy()

print(f"Dataset de Rendimento por Município — bruto: {df_rend.shape}")
print(f"  Municípios únicos: {df_rend['CO_MUNICIPIO'].nunique()}")
print(f"  UFs: {df_rend['SG_UF'].nunique()}")
print(f"  Localizações: {df_rend['NO_CATEGORIA'].str.strip().unique().tolist()}")
print(f"  Dependências: {df_rend['NO_DEPENDENCIA'].str.strip().unique().tolist()}")

Dataset de Rendimento por Município — bruto: (65594, 61)
  Municípios únicos: 5570
  UFs: 27
  Localizações: ['Total', 'Urbana', 'Rural']
  Dependências: ['Total', 'Estadual', 'Municipal', 'Pública', 'Federal', 'Privada']


In [7]:
# ── Dataset 2: Microdados — inspecionar sem carregar tudo ────────────────
df_head = pd.read_csv(
    'data/raw/microdados_ed_basica_2024.csv',
    sep=';', encoding='latin-1', nrows=5
)
print(f"Dataset 2 (microdados) — colunas totais: {df_head.shape[1]}")
print(f"  Colunas selecionadas para uso: 23 de {df_head.shape[1]}")

# Contar linhas sem carregar o arquivo todo
import subprocess
result = subprocess.run(['wc', '-l', 'data/raw/microdados_ed_basica_2024.csv'],
                       capture_output=True, text=True)
n_linhas = int(result.stdout.strip().split()[0]) - 1  # -1 header
print(f"  Total de escolas no arquivo: {n_linhas:,}")
print(f"  Dependências codificadas: 1=Federal, 2=Estadual, 3=Municipal, 4=Privada")
print(f"  Localização codificada: 1=Urbana, 2=Rural")

Dataset 2 (microdados) — colunas totais: 426
  Colunas selecionadas para uso: 23 de 426
  Total de escolas no arquivo: 215,545
  Dependências codificadas: 1=Federal, 2=Estadual, 3=Municipal, 4=Privada
  Localização codificada: 1=Urbana, 2=Rural


## 11. Limitações da Base

| # | Limitação | Impacto na Modelagem |
|---|-----------|----------------------|
| 1 | Granularidade agregada — não por escola ou aluno individual | Modelo classifica contextos municipais, não indivíduos |
| 2 | Ano único (2024) — sem série temporal | Não captura tendências; aceitável para classificação cross-sectional |
| 3 | Combinações raras com poucas escolas (ex.: Federal Rural) | Missing em ~15–20% das features de infraestrutura — imputado por mediana no NB4 |
| 4 | **Ensino Médio ausente em muitos municípios** | `reprov_med_total` e séries do EM têm ~41% de NaN — imputação é obrigatória |
| 5 | Municípios sem Ensino Médio próprio não têm como distinguir risco do EM | Limitação real: ~41% das instâncias não têm dados de EM |
| 6 | Ausência de dados socioeconômicos (renda, IDH, IDEB) | Foco do projeto é infraestrutura e contexto escolar — limitação consciente |